In [2]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm

import cv2


In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with new superset classes

In [4]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df = index_df[index_df['session'].str.startswith('vqa')].reset_index(drop=True)
print(index_df.shape)

index_df['cam'] = 'cam1'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(177, 4)
(10, 4)
(10, 5)


,video_id,template,placeholders,session,cam
0,261027-131511,Moving [something] across a surface,wallet,vqa_wallet_clean,cam1
1,261027-131534,Moving [something] across a surface until it falls down,wallet,vqa_wallet_clean,cam1


In [5]:
index_df.template.value_counts()

template
Moving [something] across a surface                              2
Moving [something] across a surface until it falls down          2
Moving [something] across a surface until it hits [something]    2
Roll [something] on table and stop it with hand                  2
Tearing [something] and throwing it in the bin                   2
Name: count, dtype: int64

### select videos

In [6]:
## special session
index_df['session_special'] = np.where(index_df.session.isin(['plug_2','magnet_2','lady_2','books_2']), 'distractors_in_video',
                               np.where(index_df.session.isin(['plug_1','magnet_1','lady_1','books_1','paper_1']), 'session_1_control', 
                                        np.where(index_df.session.isin(['plugging_1','showing_photo_of_1','attaching_1','piling_1']), 'session2_specialActions_commonobjects',
                                                 np.where(index_df['session'].str.startswith('endProduct'), 'distractors_are_endproducts',
                                                    np.where(index_df['session'].str.startswith('vqa'), 'vqa',
                                                        np.where(index_df['session'].str.startswith('mime'), 'miming_videos', 
                                                              'mime_actions_control_videos'))))))

index_df.tail(2)

,video_id,template,placeholders,session,cam,session_special
8,261027-134525,Roll [something] on table and stop it with hand,board marker,vqa_board_marker,cam1,vqa
9,261027-134559,Tearing [something] and throwing it in the bin,ball,vqa_wrong_label_not_tearing,cam1,vqa


In [7]:
for i, row in index_df[index_df.session_special.isin(['vqa'])].iterrows():
    print(f"'{row['video_id']}', # {row['session_special']} | {row['placeholders']} --> {row['template']:15} ")


'261027-131511', # vqa | wallet --> Moving [something] across a surface 
'261027-131534', # vqa | wallet --> Moving [something] across a surface until it falls down 
'261027-132224', # vqa | wallet, books --> Moving [something] across a surface until it hits [something] 
'261027-132321', # vqa | wallet --> Moving [something] across a surface 
'261027-132344', # vqa | wallet --> Moving [something] across a surface until it falls down 
'261027-132411', # vqa | wallet, books --> Moving [something] across a surface until it hits [something] 
'261027-132808', # vqa | ball --> Roll [something] on table and stop it with hand 
'261027-133237', # vqa | paper --> Tearing [something] and throwing it in the bin 
'261027-134525', # vqa | board marker --> Roll [something] on table and stop it with hand 
'261027-134559', # vqa | ball --> Tearing [something] and throwing it in the bin 


In [8]:
vids_list = [
    '261027-131511', # vqa | wallet --> Moving [something] across a surface 
    '261027-131534', # vqa | wallet --> Moving [something] across a surface until it falls down 
    '261027-132224', # vqa | wallet, books --> Moving [something] across a surface until it hits [something] 
    '261027-132321', # vqa | wallet --> Moving [something] across a surface 
    '261027-132344', # vqa | wallet --> Moving [something] across a surface until it falls down 
    '261027-132411', # vqa | wallet, books --> Moving [something] across a surface until it hits [something] 
    '261027-132808', # vqa | ball --> Roll [something] on table and stop it with hand 
    '261027-133237', # vqa | paper --> Tearing [something] and throwing it in the bin 
    '261027-134525', # vqa | board marker --> Roll [something] on table and stop it with hand 
    '261027-134559', # vqa | ball --> Tearing [something] and throwing it in the bin
]

In [13]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation for VQA dataset

- 2 cam views important to classify action. cam0 and cam1.

#### Concatenate the 2 views per video carefully

- Because glm-5.3-Flash cannot ingest 2 separate videos at the same time

In [14]:
def concatenate_videos_side_by_side(video_path_0, video_path_1, output_path):
    """Stitches two synchronized videos horizontally into a single video file."""
    cap0 = cv2.VideoCapture(video_path_0)
    cap1 = cv2.VideoCapture(video_path_1)

    fps = cap0.get(cv2.CAP_PROP_FPS) or 30.0
    width0 = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    height0 = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    height1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Match height of video 1 to video 0 if they differ
    target_height = height0

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    # Combined width = width of cam0 + width of cam1
    out_width = width0 + int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH) * (target_height / height1))
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_width, target_height))

    while cap0.isOpened() and cap1.isOpened():
        ret0, frame0 = cap0.read()
        ret1, frame1 = cap1.read()

        if not ret0 or not ret1:
            break

        # Resize frame1 if heights don't match exactly
        if frame1.shape[0] != target_height:
            new_w = int(frame1.shape[1] * (target_height / frame1.shape[0]))
            frame1 = cv2.resize(frame1, (new_w, target_height))

        # Concatenate horizontally (Left: View 0 | Right: View 1)
        combined_frame = cv2.hconcat([frame0, frame1])
        out.write(combined_frame)

    cap0.release()
    cap1.release()
    out.release()

    print(f"[*] New file created: {output_path}")

In [18]:
# File paths
videos_file_path = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw/'
combined_videos_file_path = 'data/combined_videos/'

for video_name in vids_list:
    path_cam0 = f"{videos_file_path}{video_name}_cam0.avi"
    path_cam1 = f"{videos_file_path}{video_name}_cam1.avi"
    combined_path = f"{combined_videos_file_path}{video_name}_combined.avi"

    # Stitch the videos horizontally
    concatenate_videos_side_by_side(path_cam0, path_cam1, combined_path)

[*] New file created: data/combined_videos/261027-131534_combined.avi
[*] New file created: data/combined_videos/261027-132224_combined.avi
[*] New file created: data/combined_videos/261027-132321_combined.avi
[*] New file created: data/combined_videos/261027-132344_combined.avi
[*] New file created: data/combined_videos/261027-132411_combined.avi
[*] New file created: data/combined_videos/261027-132808_combined.avi
[*] New file created: data/combined_videos/261027-133237_combined.avi
[*] New file created: data/combined_videos/261027-134525_combined.avi
[*] New file created: data/combined_videos/261027-134559_combined.avi


In [ ]:
## Cell above took ~13mins (for 10 videos)

### Run one video

In [19]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GWDG_OPENAI_API_KEY"), 
    base_url="https://chat-ai.academiccloud.de/v1"
    )
client

In [20]:
labels_list = index_df.template.unique().tolist()
labels_list.sort()

print(labels_list)

['Moving [something] across a surface', 'Moving [something] across a surface until it falls down', 'Moving [something] across a surface until it hits [something]', 'Roll [something] on table and stop it with hand', 'Tearing [something] and throwing it in the bin']


In [21]:

# Update prompt to inform model of the split layout
system_prompt = (
    "You are an expert multi-view video action classification model. "
    "You are provided with a single video frame containing two side-by-side synchronized camera views of the exact same temporal event: "
    "the LEFT half is 'View 0' and the RIGHT half is 'View 1'. "
    "You must synthesize spatial and temporal information across both views to classify the primary action."
)

user_prompt = f"""
Analyze this synchronized multi-view video containing two camera angles (LEFT side: View 0, RIGHT side: View 1) of the exact same temporal event and classify the action performed.

Candidate Action Classes:
{json.dumps(labels_list, indent=2)}

Instructions:
1. Synthesize the sequence of events across BOTH camera halves (Left = View 0, Right = View 1). A single view cannot determine the true class.
2. Assign a probability (float between 0.0 and 1.0) to EVERY class. The probabilities MUST sum to 1.0.
3. Select the top predicted class name based on the combined multi-view evidence.
4. Provide a brief visual observation for View 0 (Left), View 1 (Right), and how they temporally connect.

Output strictly in JSON matching this schema:
{{
  "view_0_observation": "<1-sentence visual description of View 0 (Left side)>",
  "view_1_observation": "<1-sentence visual description of View 1 (Right side)>",
  "combined_reasoning": "<1-sentence explanation of how both views complement each other to reveal the true action>",
  "predicted_class": "<exact class name from list>",
  "probabilities": {{
    "{labels_list[0]}": 0.0,
    "{labels_list[1]}": 0.0,
    "...": 0.0
  }}
}}
"""

print(user_prompt)



Analyze this synchronized multi-view video containing two camera angles (LEFT side: View 0, RIGHT side: View 1) of the exact same temporal event and classify the action performed.

Candidate Action Classes:
[
  "Moving [something] across a surface",
  "Moving [something] across a surface until it falls down",
  "Moving [something] across a surface until it hits [something]",
  "Roll [something] on table and stop it with hand",
  "Tearing [something] and throwing it in the bin"
]

Instructions:
1. Synthesize the sequence of events across BOTH camera halves (Left = View 0, Right = View 1). A single view cannot determine the true class.
2. Assign a probability (float between 0.0 and 1.0) to EVERY class. The probabilities MUST sum to 1.0.
3. Select the top predicted class name based on the combined multi-view evidence.
4. Provide a brief visual observation for View 0 (Left), View 1 (Right), and how they temporally connect.

Output strictly in JSON matching this schema:
{
  "view_0_observa

In [23]:
# Encode combined video to base64
video_name = vids_list[0]

combined_path = f'data/combined_videos/{video_name}_combined.avi'

with open(combined_path, "rb") as f:
    b64_combined_video = base64.b64encode(f.read()).decode("utf-8")


In [24]:
start_time = time.time()

response = client.chat.completions.create(
    model="glm-5.3-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "video_url",
                    "video_url": {"url": f"data:video/avi;base64,{b64_combined_video}"},
                },
            ],
        },
    ],
    response_format={"type": "json_object"},
    temperature=0.1,
)

elapsed_time = round(time.time() - start_time, 2)

print(f"Elapsed time: {elapsed_time}s")

Elapsed time: 25.53s


In [25]:
prediction_json = json.loads(response.choices[0].message.content)
prediction_json.keys()

dict_keys(['view_0_observation', 'view_1_observation', 'combined_reasoning', 'predicted_class', 'probabilities'])

In [26]:
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

# 3. Update the dictionary to log both video file names
run_result = {
    "video_file": os.path.basename(combined_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "predicted_class": prediction_json.get("predicted_class"),
    "view_0_observation": prediction_json.get("view_0_observation"),
    "view_1_observation": prediction_json.get("view_1_observation"),
    "combined_reasoning": prediction_json.get("combined_reasoning"),
    "probabilities": prediction_json.get("probabilities", {}),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/action_predictions_vqa_2views.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "261027-131511_combined.avi",
  "model": "glm-5.3-flash",
  "elapsed_seconds": 25.53,
  "prompt_tokens": 10042,
  "completion_tokens": 784,
  "total_tokens": 10826,
  "predicted_class": "Moving [something] across a surface until it falls down",
  "view_0_observation": "A hand pushes a brown wallet flat across the light wooden table from the left side, with the wallet sliding rightward and eventually exiting the frame.",
  "view_1_observation": "From a side angle showing the table edge, the same hand pushes the wallet steadily toward the table's far edge, where it sits precariously at the boundary by the final frame.",
  "combined_reasoning": "View 0 shows the wallet being slid across the surface while View 1 reveals the critical endpoint\u2014the wallet is pushed all the way to the table edge and is about to drop off, confirming the action continues until the object falls.",
  "probabilities": {
    "Moving [something] across a surface": 0.15,
    "Moving [something] 

## For loop over many videos

In [28]:
# Load environment variables from .env
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GWDG_OPENAI_API_KEY"), 
    base_url="https://chat-ai.academiccloud.de/v1"
    )

In [29]:

# Update prompt to inform model of the split layout
system_prompt = (
    "You are an expert multi-view video action classification model. "
    "You are provided with a single video frame containing two side-by-side synchronized camera views of the exact same temporal event: "
    "the LEFT half is 'View 0' and the RIGHT half is 'View 1'. "
    "You must synthesize spatial and temporal information across both views to classify the primary action."
)

user_prompt = f"""
Analyze this synchronized multi-view video containing two camera angles (LEFT side: View 0, RIGHT side: View 1) of the exact same temporal event and classify the action performed.

Candidate Action Classes:
{json.dumps(labels_list, indent=2)}

Instructions:
1. Synthesize the sequence of events across BOTH camera halves (Left = View 0, Right = View 1). A single view cannot determine the true class.
2. Assign a probability (float between 0.0 and 1.0) to EVERY class. The probabilities MUST sum to 1.0.
3. Select the top predicted class name based on the combined multi-view evidence.
4. Provide a brief visual observation for View 0 (Left), View 1 (Right), and how they temporally connect.

Output strictly in JSON matching this schema:
{{
  "view_0_observation": "<1-sentence visual description of View 0 (Left side)>",
  "view_1_observation": "<1-sentence visual description of View 1 (Right side)>",
  "combined_reasoning": "<1-sentence explanation of how both views complement each other to reveal the true action>",
  "predicted_class": "<exact class name from list>",
  "probabilities": {{
    "{labels_list[0]}": 0.0,
    "{labels_list[1]}": 0.0,
    "...": 0.0
  }}
}}
"""


videos_dir = 'data/combined_videos/'
video_paths = [videos_dir + video + '_combined.avi' for video in vids_list[1:]]
video_paths

['data/combined_videos/261027-131534_combined.avi',
 'data/combined_videos/261027-132224_combined.avi',
 'data/combined_videos/261027-132321_combined.avi',
 'data/combined_videos/261027-132344_combined.avi',
 'data/combined_videos/261027-132411_combined.avi',
 'data/combined_videos/261027-132808_combined.avi',
 'data/combined_videos/261027-133237_combined.avi',
 'data/combined_videos/261027-134525_combined.avi',
 'data/combined_videos/261027-134559_combined.avi']

In [ ]:
from tqdm import tqdm

pbar = tqdm(video_paths, desc="Processing Videos")

for video_path in pbar:
    # Load video in b64
    with open(video_path, "rb") as f:
        b64_combined_video = base64.b64encode(f.read()).decode("utf-8")
    
    # API call
    start_time = time.time()

    response = client.chat.completions.create(
        model="glm-5.3-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": user_prompt},
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/avi;base64,{b64_combined_video}"},
                    },
                ],
            },
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )

    elapsed_time = round(time.time() - start_time, 2)


    ## Parse Output & Save data
    # Extract JSON prediction content
    prediction_json = json.loads(response.choices[0].message.content)

    # Extract token & performance metadata
    usage = response.usage

    # 3. Update the dictionary to log both video file names
    run_result = {
        "video_file": os.path.basename(combined_path),
        "model": response.model,
        "elapsed_seconds": elapsed_time,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "completion_tokens": usage.completion_tokens if usage else None,
        "total_tokens": usage.total_tokens if usage else None,
        "predicted_class": prediction_json.get("predicted_class"),
        "view_0_observation": prediction_json.get("view_0_observation"),
        "view_1_observation": prediction_json.get("view_1_observation"),
        "combined_reasoning": prediction_json.get("combined_reasoning"),
        "probabilities": prediction_json.get("probabilities", {}),
    }

    # Append result as a single JSON line
    with open('data/vlm_eval_results/action_predictions_vqa_2views.jsonl', "a", encoding="utf-8") as f:
        f.write(json.dumps(run_result) + "\n")



Processing Videos:   0%|          | 0/9 [00:00<?, ?it/s]

# Loading and analyzing the jsonl outputs of the VLM

In [14]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/action_predictions_normal.jsonl", lines=True)

# 2. Expand the nested 'probabilities' dictionary into clean individual columns
probs_df = pd.json_normalize(df["probabilities"])

# 3. Merge metadata with probability columns
df = pd.concat([df.drop(columns=["probabilities"]), probs_df], axis=1)
print(df.shape)
print(df.head(2))

(10, 17)
               video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                       predicted_class                                                                                                                            reasoning  Attaching [something] to [something]  Bending [something]  Moving [something]  Moving [something] and [something]  Pretending to do something without touching objects  Putting [something] somewhere  Showing or exhibiting [something]  Stacking things  Tearing [something]
0  260922-134333_cam1.avi  glm-5.3-flash             7.53           5074                332          5406  Attaching [something] to [something]  A person holds a power strip with one hand and inserts a black plug into one of its sockets, attaching the plug to the power strip.                                  0.85                  0.0                0.05                                0.03                                                  0

In [15]:
index_df['video_file'] = index_df.video_id + '_cam1.avi'

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(10, 25)
               video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                       predicted_class                                                                                                                                                            reasoning  Attaching [something] to [something]  Bending [something]  ...  Stacking things  Tearing [something]       video_id                               template                    placeholders      session   cam label                     template_superset                        session_special
9  260930-145329_cam1.avi  glm-5.3-flash            16.43           6276                397          6673  Attaching [something] to [something]  A person picks up a red marker and its cap with both hands and presses the cap onto the marker tip, clearly attaching the cap to the marker before setting it down.                                  0.95                  0.0  ...              0.0   

In [16]:
df['correct_top1'] = df['predicted_class'] == df['template_superset']

print(f'Examples: {df.shape[0]}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 10
top1acc:  1.0


In [17]:
import textwrap

pd.set_option('display.float_format',  '{:,.4f}'.format)

def analyze_df(df):
    """ Needs labels_list in addition to df """

    print('='*20, 'df stats by session', '='*20)

    r = df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1acc=('correct_top1', 'mean')
    )

    print(r)

    print('='*20, 'Individual Videos vy session_special', '='*20)

    for session in df.session_special.unique():
        print('-'*20, session, '-'*20)

        for i, row in df[df['session_special']==session].iterrows():
            print(row[['video_id', 'session', 'placeholders','template_superset', 'predicted_class', 'correct_top1']].to_frame().T)
            print('\n', "Probability Distribution:", '\n\n', row[labels_list])
            print(f'Sum of all probabilities                                {row[labels_list].sum():.3f}')            
            print('\nVLM reasoning:', textwrap.fill(row['reasoning'], width=100))
            print('-'*60)


In [18]:
analyze_df(df)

==================== df stats by session ====================
             videos  top1acc
session                     
attaching_1       1   1.0000
books_1           1   1.0000
lady_1            1   1.0000
magnet_1          1   1.0000
paper_1           1   1.0000
plug_1            3   1.0000
plugging_1        2   1.0000
==================== Individual Videos vy session_special ====================
-------------------- session_1_control --------------------
        video_id session                placeholders                     template_superset                       predicted_class correct_top1
0  260922-134333  plug_1  plug, electrical extension  Attaching [something] to [something]  Attaching [something] to [something]         True

 Probability Distribution: 

 Attaching [something] to [something]                  0.8500
Bending [something]                                   0.0000
Moving [something]                                    0.0500
Moving [something] and [something]      

In [19]:
## Normal/control videos. 11 tested. All have very good prob dists and VLM's reasoning for its answer is always good. (Probs dont sum to 1.0 in few cases but still >=0.98 for these cases.)

In [20]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/action_predictions_distractors.jsonl", lines=True)

# 2. Expand the nested 'probabilities' dictionary into clean individual columns
probs_df = pd.json_normalize(df["probabilities"])

# 3. Merge metadata with probability columns
df = pd.concat([df.drop(columns=["probabilities"]), probs_df], axis=1)
print(df.shape)
print(df.head(2))

(23, 17)
               video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                predicted_class                                                                                                                               reasoning  Attaching [something] to [something]  Bending [something]  Moving [something]  Moving [something] and [something]  Pretending to do something without touching objects  Putting [something] somewhere  Showing or exhibiting [something]  Stacking things  Tearing [something]
0  260922-141236_cam1.avi  glm-5.3-flash         596.6400           3872                628          4500  Putting [something] somewhere  A hand brings a plastic water bottle into view and places it down on the table next to a marker, which is a clear putting-down action.                                0.0000               0.0000              0.1500                              0.0500                                               0.0000      

In [21]:
index_df['video_file'] = index_df.video_id + '_cam1.avi'

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN

df['correct_top1'] = df['predicted_class'] == df['template_superset']

print(df.shape)
print(df.sample(2))

print(f'Examples: {df.shape[0]}')
print(f'top1acc: {df.correct_top1.mean():4}')


(23, 26)
                video_file          model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens                predicted_class                                                                                                                                                      reasoning  Attaching [something] to [something]  Bending [something]  ...  Tearing [something]       video_id                                 template                placeholders  session   cam  label              template_superset       session_special correct_top1
13  260930-122408_cam1.avi  glm-5.3-flash           4.7500           2670                494          3164  Putting [something] somewhere                                                      A hand is seen holding a black power plug and then dropping it onto a light-colored wooden table surface.                                0.0000               0.0000  ...               0.0000  260930-122408         Putting [something] on a surface

In [22]:
analyze_df(df)

==================== df stats by session ====================
                         videos  top1acc
session                                 
bending_1                     2   1.0000
books_2                       4   0.2500
lady_2                        1   1.0000
magnet_2                      2   1.0000
moveCloserToEachOther_1       1   1.0000
plug_2                        9   1.0000
pretend_pickUp_1              3   1.0000
pretend_sprinkle_1            1   1.0000
==================== Individual Videos vy session_special ====================
-------------------- distractors_in_video --------------------
        video_id session                placeholders              template_superset                predicted_class correct_top1
0  260922-141236  plug_2  water bottle, board marker  Putting [something] somewhere  Putting [something] somewhere         True

 Probability Distribution: 

 Attaching [something] to [something]                  0.0000
Bending [something]                   

In [23]:
## only 2/23 Examples showing spikes in actions related to distractors even though prediction is correct. These are anamolies though not a pattern. Both examples are where plug and electrical_extension exist and there is a spike in 'Attaching [something] to [something]'. I have other examples where these objects appear and no spike (Reasoing of the model is impeccable)
spike_examples = [
'260930-122317', 
'260930-122827'
]


## Misclassified 3/23 only: All 3 are misclassifying when i am "putting a pile of books down" but model thinks i am "stacking things". BUT reasoning is consistent.
misclassified_examples = [
'260930-143429',
'260930-143444',
'260930-143505',
]

## Overall impression is the VLM does not have object bias by and large.

In [24]:
analyze_df(df[df.video_id.isin(spike_examples)])

==================== df stats by session ====================
         videos  top1acc
session                 
plug_2        2   1.0000
==================== Individual Videos vy session_special ====================
-------------------- distractors_in_video --------------------
        video_id session                placeholders              template_superset                predicted_class correct_top1
2  260930-122317  plug_2  plug, electrical extension  Putting [something] somewhere  Putting [something] somewhere         True

 Probability Distribution: 

 Attaching [something] to [something]                  0.0800
Bending [something]                                   0.0000
Moving [something]                                    0.1500
Moving [something] and [something]                    0.0200
Pretending to do something without touching objects   0.0500
Putting [something] somewhere                         0.6000
Showing or exhibiting [something]                     0.0500
Stackin

In [25]:
analyze_df(df[df.video_id.isin(misclassified_examples)])

==================== df stats by session ====================
         videos  top1acc
session                 
books_2       3   0.0000
==================== Individual Videos vy session_special ====================
-------------------- distractors_in_video --------------------
        video_id  session placeholders              template_superset  predicted_class correct_top1
9  260930-143429  books_2        books  Putting [something] somewhere  Stacking things        False

 Probability Distribution: 

 Attaching [something] to [something]                  0.0200
Bending [something]                                   0.0000
Moving [something]                                    0.0800
Moving [something] and [something]                    0.0500
Pretending to do something without touching objects   0.0000
Putting [something] somewhere                         0.1500
Showing or exhibiting [something]                     0.0000
Stacking things                                       0.6800
Te

### To answer Felix's specific questions

In [29]:
analyze_df(df[df.placeholders.str.contains('extension')])

==================== df stats by session ====================
         videos  top1acc
session                 
plug_2        3   1.0000
==================== Individual Videos vy session_special ====================
-------------------- distractors_in_video --------------------
        video_id session                placeholders              template_superset                predicted_class correct_top1
2  260930-122317  plug_2  plug, electrical extension  Putting [something] somewhere  Putting [something] somewhere         True

 Probability Distribution: 

 Attaching [something] to [something]                  0.0800
Bending [something]                                   0.0000
Moving [something]                                    0.1500
Moving [something] and [something]                    0.0200
Pretending to do something without touching objects   0.0500
Putting [something] somewhere                         0.6000
Showing or exhibiting [something]                     0.0500
Stackin

In [ ]:
## something something something